In [30]:
from pyhectr.surface import (
    parse_xtl,
    compute_pairwise_vectors,
    find_vectors_with_target_angle_minimal_stream,
    find_shortest_normal_vector,
    validate_surface_transform,
)

import numpy as np

# Surface transformation search from VESTA cuts

This notebook documents the exploratory step used to prepare candidate rectangular surface cell transformations before generating the final high precision cells with `pyhectr.surface`.

The practical workflow is:

1. Build or expand the bcc Nb bulk cell in VESTA.
2. Cut the candidate high index surface, for example Nb(326), so that the visible surface plane is the intended `(hkl)` plane.
3. Export the cut or expanded structure as `.xtl`.
4. Search the exported atom coordinates for two short in-plane translation vectors that are perpendicular to each other and whose cross product is collinear with the candidate surface normal.
5. Choose a third vector parallel to the outward normal `[h,k,l]`.
6. Assemble the integer transformation matrix `P` with these three vectors as columns, choose signs so that `det(P) > 0`, and `hkl @ P = [0, 0, positive]`.
7. Copy the validated matrix into the surface cell generation notebook, where the final coordinates are generated directly from the integer transformation.

The VESTA step is useful for visual intuition and for finding candidate repeat vectors. It should not be the final source of ROD coordinates for large high index cells, because rounded exported fractional coordinates can produce numerical artifacts in the structure factor calculation.

For cubic Nb, fractional coordinate differences can be interpreted as direct lattice vector coefficients, so ordinary dot products are sufficient for this search. For a non cubic lattice, convert coordinates to Cartesian space and use the reciprocal lattice normal.


# Transformation matrix search

## Nb(326)

The target surface plane is `(3,2,6)`. The outward normal is taken as `[3,2,6]`, which is valid here because the parent Nb cell is cubic.

The search returns two in-plane vectors. The signs used in the transformation matrix are then chosen manually so that the final matrix is right-handed and the third column points toward `+hkl`.


In [2]:
filename = "nb_3_2_6_plane.xtl"
atoms = parse_xtl(filename)
c = np.array([3,2,6])
len(atoms)

4933

In [3]:
pairs, vecs = compute_pairwise_vectors(atoms)

match = find_vectors_with_target_angle_minimal_stream(
            pairs, vecs,
            target_angle=90.0,
            tol_angle=1e-6,
            surface_normal=c,
            surface_tol=1e-6,
            block=100_000)           # tune for your RAM

best_area = 49.0
best_global_i = 197, best_global_j = 20341


best_area = 35.0
best_global_i = 180, best_global_j = 245505




In [4]:
match

[((0, 181),
  (50, 181),
  array([ 2.,  0., -1.]),
  array([  2., -15.,   4.]),
  90.0,
  35.0)]

In [5]:
a, b = np.array(match[0][2:4])

M = np.array([a,-b,c]).T
M

array([[ 2., -2.,  3.],
       [ 0., 15.,  2.],
       [-1., -4.,  6.]])

In [6]:
np.linalg.det(M)

244.9999999999999

In [7]:
if match:
    for pair1, pair2, vec1, vec2, angle, area in match:
        print(f"Between atoms {pair1} with vector {vec1} and atoms {pair2} with vector {vec2}:")
        print(f"    Angle = {angle:.2f}°, Area = {area:.2f}")

Between atoms (0, 181) with vector [ 2.  0. -1.] and atoms (50, 181) with vector [  2. -15.   4.]:
    Angle = 90.00°, Area = 35.00


## Nb(438)

This candidate corresponds to the EBSD orientation after using a larger nearest-`hkl` search range. The same search logic is used: find two orthogonal in-plane repeat vectors and combine them with the outward normal `[4,3,8]`.


In [10]:
filename = "nb_4_3_8_plane.xtl"
atoms = parse_xtl(filename)
c = np.array([4,3,8])
len(atoms)

2231

In [11]:
pairs, vecs = compute_pairwise_vectors(atoms)

match = find_vectors_with_target_angle_minimal_stream(
            pairs, vecs,
            target_angle=90.0,
            tol_angle=1e-6,
            surface_normal=c,
            surface_tol=1e-6,
            block=100_000)           # tune for your RAM


best_area = 47.16990566028302
best_global_i = 52, best_global_j = 26760




In [12]:
match

[((0, 53),
  (12, 79),
  array([ 2.,  0., -1.]),
  array([  3., -20.,   6.]),
  90.0,
  47.16990566028302)]

In [13]:
a, b = np.array(match[0][2:4])

In [14]:
M = np.array([-a,b,c]).T
M

array([[ -2.,   3.,   4.],
       [ -0., -20.,   3.],
       [  1.,   6.,   8.]])

In [15]:
np.linalg.det(M)

444.99999999999994

In [16]:
if match:
    for pair1, pair2, vec1, vec2, angle, area in match:
        print(f"Between atoms {pair1} with vector {vec1} and atoms {pair2} with vector {vec2}:")
        print(f"    Angle = {angle:.2f}°, Area = {area:.2f}")

Between atoms (0, 53) with vector [ 2.  0. -1.] and atoms (12, 79) with vector [  3. -20.   6.]:
    Angle = 90.00°, Area = 47.17


## Nb(539)

This candidate comes from the XRD/GrainSpotter orientation and was used for the main ROD refinement. The final matrix should satisfy `hkl @ P = [0, 0, positive]` for `hkl = (5,3,9)` and should have a positive determinant.


In [19]:
filename = "nb_5_3_9_plane.xtl"
atoms = parse_xtl(filename)
c = np.array([5,3,9])
len(atoms)

560

In [20]:
len(atoms)

560

In [21]:
pairs, vecs = compute_pairwise_vectors(atoms)

match = find_vectors_with_target_angle_minimal_stream(
            pairs, vecs,
            target_angle=90.0,
            tol_angle=1e-6,
            surface_normal=c,
            surface_tol=1e-6,
            block=100_000)           # tune for your RAM


best_area = 21.447610589527216
best_global_i = 1, best_global_j = 641




In [22]:
match

[((0, 2),
  (1, 84),
  array([ 0.,  3., -1.]),
  array([ 6., -1., -3.]),
  90.0,
  21.447610589527216)]

In [23]:
a, b = np.array(match[0][2:4])

In [24]:
M = np.array([-a,b,c]).T
M

array([[-0.,  6.,  5.],
       [-3., -1.,  3.],
       [ 1., -3.,  9.]])

In [25]:
np.linalg.det(M)

229.99999999999983

In [26]:
if match:
    for pair1, pair2, vec1, vec2, angle, area in match:
        print(f"Between atoms {pair1} with vector {vec1} and atoms {pair2} with vector {vec2}:")
        print(f"    Angle = {angle:.2f}°, Area = {area:.2f}")

Between atoms (0, 2) with vector [ 0.  3. -1.] and atoms (1, 84) with vector [ 6. -1. -3.]:
    Angle = 90.00°, Area = 21.45


In [31]:
P = np.rint(M).astype(int)
print("det(P) =", np.linalg.det(P))
validate_surface_transform(P, c)
print("hkl @ P =", c @ P)


det(P) = 229.99999999999983
hkl @ P = [  0   0 115]


## Nb(6 4 11)

This candidate comes from the XRD/GrainSpotter orientation and was next precision step after `hkl = (5,3,9)`. The final matrix should satisfy `hkl @ P = [0, 0, positive]` for `hkl = (6,4,11)` and should have a positive determinant.


In [32]:
filename = "nb_6_4_11_plane_reduced3.xtl"
atoms = parse_xtl(filename)
c = np.array([5,3,9])
len(atoms)

2180

In [33]:
len(atoms)

2180

In [34]:
pairs, vecs = compute_pairwise_vectors(atoms)

match = find_vectors_with_target_angle_minimal_stream(
            pairs, vecs,
            target_angle=90.0,
            tol_angle=1e-6,
            surface_normal=c,
            surface_tol=1e-6,
            block=100_000)           # tune for your RAM


best_area = 21.447610589527216
best_global_i = 2, best_global_j = 181




In [35]:
match

[((0, 3),
  (0, 182),
  array([ 0.,  3., -1.]),
  array([ 6., -1., -3.]),
  90.0,
  21.447610589527216)]

In [36]:
a, b = np.array(match[0][2:4])

In [37]:
M = np.array([-a,b,c]).T
M

array([[-0.,  6.,  5.],
       [-3., -1.,  3.],
       [ 1., -3.,  9.]])

In [38]:
np.linalg.det(M)

229.99999999999983

In [39]:
if match:
    for pair1, pair2, vec1, vec2, angle, area in match:
        print(f"Between atoms {pair1} with vector {vec1} and atoms {pair2} with vector {vec2}:")
        print(f"    Angle = {angle:.2f}°, Area = {area:.2f}")

Between atoms (0, 3) with vector [ 0.  3. -1.] and atoms (0, 182) with vector [ 6. -1. -3.]:
    Angle = 90.00°, Area = 21.45


In [40]:
P = np.rint(M).astype(int)
print("det(P) =", np.linalg.det(P))
validate_surface_transform(P, c)
print("hkl @ P =", c @ P)


det(P) = 229.99999999999983
hkl @ P = [  0   0 115]
